# Pipeline 5 : Predire la puissance exacte d'une molecule

La question du chercheur : savoir qu'une molecule est active c'est bien, mais quelle sera precisement sa puissance d'inhibition ?

La classification du notebook precedent repond par oui ou non. Ici on veut un chiffre : le pIC50, c'est-a-dire la puissance continue de la molecule. Cette information est plus fine et plus utile pour prioriser. Entre deux molecules toutes deux actives, celle dont le pIC50 predit est le plus eleve sera testee en premier. C'est un probleme de regression, exactement comme predire la valeur marchande d'un joueur, mais ici on predit la puissance d'une molecule.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib

C_BLEU = "#1f6f8b"
C_ORANGE = "#e0771a"
C_VERT = "#2e8b57"
C_ROUGE = "#9b2226"
C_GRIS = "#8d99ae"

plt.rcParams.update({
    "figure.figsize": (13, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 13
})

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('egfr_descripteurs.csv')
fingerprints = np.load('egfr_fingerprints.npy')

descripteurs = ['poids_moleculaire', 'logP', 'donneurs_H', 'accepteurs_H',
                'tpsa', 'liaisons_rotatives', 'anneaux_aromatiques']

# On combine descripteurs standardises et empreintes, la representation la plus riche
X_desc = StandardScaler().fit_transform(df[descripteurs].values)
X = np.hstack([X_desc, fingerprints])
y = df['pIC50'].values

print(f"Molecules : {len(df)} | Dimensions des features : {X.shape[1]}")
print(f"pIC50 : min {y.min():.2f}, moyenne {y.mean():.2f}, max {y.max():.2f}")

# 1. Modelisation : baseline puis modeles avances

On suit la meme discipline que partout : une regression lineaire en baseline, puis Random Forest et XGBoost. La validation croisee en cinq plis donne des estimations honnetes qui ne dependent pas d'un decoupage chanceux.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

modeles = {
    'Regression Lineaire': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                            random_state=RANDOM_STATE)
}

resultats = []
for nom, modele in modeles.items():
    rmse_cv = -cross_val_score(modele, X_train, y_train, cv=kf,
                               scoring='neg_root_mean_squared_error').mean()
    r2_cv = cross_val_score(modele, X_train, y_train, cv=kf, scoring='r2').mean()
    resultats.append({'Modele': nom, 'RMSE_cv': rmse_cv, 'R2_cv': r2_cv})
    print(f"{nom:22s} | RMSE (cv) : {rmse_cv:.3f} | R2 (cv) : {r2_cv:.3f}")

gc.collect()

In [ ]:
# Entrainement final du meilleur modele et evaluation sur le test
df_res = pd.DataFrame(resultats)
meilleur_nom = df_res.loc[df_res['R2_cv'].idxmax(), 'Modele']
meilleur = modeles[meilleur_nom]
meilleur.fit(X_train, y_train)
y_pred = meilleur.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Meilleur modele : {meilleur_nom}")
print(f"Sur le jeu de test : RMSE {rmse:.3f} | MAE {mae:.3f} | R2 {r2:.3f}")
print()
print(f"Traduction concrete : une MAE de {mae:.2f} sur le pIC50 signifie que le modele se trompe")
print(f"en moyenne de {mae:.2f} unite de pIC50, soit environ un facteur {10**mae:.1f} sur l'IC50 reel.")
print("Autrement dit, le modele situe la puissance a moins d'un ordre de grandeur pres, ce qui est")
print("tres correct pour un premier tri de molecules mais insuffisant pour un classement fin entre")
print("candidates tres proches.")

# 2. Le graphique signature : predit contre reel

In [ ]:
plt.figure(figsize=(9, 8))
plt.scatter(y_test, y_pred, alpha=0.5, s=35, color=C_BLEU, edgecolors='white', linewidth=0.3)
lims = [min(y_test.min(), y_pred.min()) - 0.5, max(y_test.max(), y_pred.max()) + 0.5]
plt.plot(lims, lims, color=C_ORANGE, linestyle='--', linewidth=2, label='Prediction parfaite')
plt.xlabel("pIC50 reel (mesure en laboratoire)")
plt.ylabel("pIC50 predit par le modele")
plt.title(f"Puissance predite contre puissance reelle ({meilleur_nom}, R2 = {r2:.2f})")
plt.legend()
plt.tight_layout()
plt.show()

print("Chaque point est une molecule du jeu de test. Plus les points collent a la diagonale orange, meilleures sont les predictions. On observe generalement un nuage assez bien aligne mais qui s'evase, ce qui est typique de la prediction d'activite : le modele attrape la tendance generale, distingue clairement les molecules faibles des puissantes, mais peine sur les nuances fines. Pour un chercheur, cela veut dire qu'on peut faire confiance au classement grossier mais pas a une difference de pIC50 de quelques dixiemes.")

# 3. Analyse des residus

In [ ]:
residus = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(residus, bins=40, kde=True, color=C_BLEU, ax=axes[0])
axes[0].axvline(0, color=C_ORANGE, linestyle='--', linewidth=1.5)
axes[0].set_title("Distribution des erreurs de prediction")
axes[0].set_xlabel("Erreur (pIC50 reel moins predit)")

axes[1].scatter(y_pred, residus, alpha=0.4, s=25, color=C_BLEU, edgecolors='none')
axes[1].axhline(0, color=C_ORANGE, linestyle='--', linewidth=1.5)
axes[1].set_title("Erreurs en fonction de la valeur predite")
axes[1].set_xlabel("pIC50 predit")
axes[1].set_ylabel("Erreur")

plt.tight_layout()
plt.show()

print("Les erreurs forment une cloche centree sur zero, donc le modele ne surestime ni ne sous-estime systematiquement, c'est rassurant. Sur le graphique de droite, on verifie que les erreurs ne dependent pas trop de la valeur predite. Si on voyait les erreurs exploser aux extremites, cela signalerait que le modele est peu fiable sur les molecules tres faibles ou tres puissantes, ce qui est justement la ou on aimerait qu'il soit bon.")

# 4. Deploiement

In [ ]:
joblib.dump({'modele': meilleur, 'descripteurs': descripteurs},
            'modele_regression_pic50.pkl')
print("Modele sauvegarde : modele_regression_pic50.pkl")
print(f"Ce modele estimera la puissance d'inhibition d'une nouvelle molecule dans le dashboard.")

# Conclusion

Le modele de regression estime la puissance d'inhibition d'une molecule a moins d'un ordre de grandeur pres, ce qui est utile pour un premier classement mais qu'il faut presenter avec honnetete. On ne predit pas l'IC50 au nanomolaire pres, et personne dans le domaine ne le pretend.

La performance plafonne pour deux raisons qu'on assume. D'abord l'incertitude experimentale des mesures d'origine fixe une limite infranchissable. Ensuite, deux molecules aux empreintes tres similaires peuvent avoir des activites tres differentes a cause d'un detail structurel, ce qu'on appelle un activity cliff, un phenomene qu'aucun modele classique ne capture bien et qu'on ira justement examiner dans le notebook bonus. La regression reste un excellent outil de priorisation, a condition de l'utiliser pour trier et non pour affirmer une valeur exacte.